# 03. Intelektualieji metodai: SVM ir MLP (UCI HAR)

Šiame etape **nedarome** SOM, abliacijos ir robustness bandymų. Tikslas — tomis pačiomis sąlygomis įgyvendinti ir palyginti **linijinį daugiaklasį SVM** ir **MLP**.

Naudojamas **tas pats** 1 etapo subject-disjoint skaidymas (`results/subject_disjoint_split.npz`). Naujo TRAIN/TEST nekuriame. Baseline rezultatų neperskaičiuojame — juos tik užkrauname iš `results/baseline_results.csv`.

## Bendros palyginimo sąlygos

Abiejų modelių hiperparametrai parenkami **tik TRAIN** su `GroupKFold(n_splits=5)` pagal `subjects_train_final`. Parinkimo metrika — Macro-F1 (`f1_macro`).

Galutinis TEST **nenaudojamas** hiperparametrų paieškai, modelio pasirinkimui, scaler `fit` ir cross-validation. TEST lieka vienam įvertinimui po to, kai parametrai jau parinkti.

`StandardScaler` abiem modeliams yra **Pipeline viduje**, kad kiekviename CV fold'e scaleris būtų mokomas tik iš tos fold'o mokymo dalies ir nebūtų data leakage.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"sklearn versija: {sklearn.__version__}")

RANDOM_STATE = 42
sklearn versija: 1.9.1


## TRAIN/TEST užkrovimas

Užkrauname tik jau išsaugotą skaidymą ir patikriname, kad TRAIN ir TEST subject sankirta vis dar tuščia.

In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = None
for cand in candidates:
    if (cand / "results").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

results_dir = project_root / "results"
split_path = results_dir / "subject_disjoint_split.npz"
baseline_path = results_dir / "baseline_results.csv"

if not split_path.exists():
    raise FileNotFoundError(
        f"Nerastas {split_path}. Pirmiausia paleiskite 01_data_preparation.ipynb."
    )
if not baseline_path.exists():
    raise FileNotFoundError(
        f"Nerastas {baseline_path}. Pirmiausia paleiskite 02_baselines.ipynb."
    )

split = np.load(split_path, allow_pickle=True)

X_train = split["X_train_final"]
X_test = split["X_test_final"]
y_train = split["y_train_final"]
y_test = split["y_test_final"]
subjects_train = split["subjects_train_final"]
subjects_test = split["subjects_test_final"]
activity_ids = split["activity_ids"]
activity_names = split["activity_names"]
saved_random_state = int(split["random_state"][0])

id_to_name = {int(i): str(n) for i, n in zip(activity_ids, activity_names)}
train_subjects = set(subjects_train.tolist())
test_subjects = set(subjects_test.tolist())
subject_overlap = sorted(train_subjects & test_subjects)

print(f"Projekto šaknis: {project_root}")
print(f"Skaidymas: {split_path}")
print(f"Išsaugotas RANDOM_STATE: {saved_random_state}")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("TRAIN subject skaičius:", len(train_subjects))
print("TEST subject skaičius:", len(test_subjects))
print("TRAIN ∩ TEST subject:", subject_overlap)
assert subject_overlap == [], "TRAIN ir TEST subject sankirta turi būti tuščia."
print("Patikra: TRAIN ir TEST subject nesikerta.")

Projekto šaknis: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26
Skaidymas: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\subject_disjoint_split.npz
Išsaugotas RANDOM_STATE: 42
X_train: (7144, 561) y_train: (7144,)
X_test: (3155, 561) y_test: (3155,)
TRAIN subject skaičius: 21
TEST subject skaičius: 9
TRAIN ∩ TEST subject: []
Patikra: TRAIN ir TEST subject nesikerta.


In [3]:
cv = GroupKFold(n_splits=5)
labels = [int(i) for i in activity_ids]
caught_warnings = []


def fit_and_log_warnings(estimator, X, y, label, **fit_params):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(X, y, **fit_params)
        unique_msgs = []
        for item in caught:
            msg = f"{item.category.__name__}: {item.message}"
            caught_warnings.append({"etapas": label, "pranesimas": msg})
            if msg not in unique_msgs:
                unique_msgs.append(msg)
                print(f"[{label}] {msg}")
        if not unique_msgs:
            print(f"[{label}] Warning nebuvo.")
    return estimator


def confusion_df(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return pd.DataFrame(
        cm,
        index=[f"tikra_{id_to_name[i]}" for i in labels],
        columns=[f"pred_{id_to_name[i]}" for i in labels],
    )


def cv_summary(grid, param_cols):
    cols = param_cols + ["mean_test_score", "std_test_score", "rank_test_score"]
    table = pd.DataFrame(grid.cv_results_)[cols].sort_values("rank_test_score")
    return table.rename(
        columns={
            "mean_test_score": "CV_MacroF1_mean",
            "std_test_score": "CV_MacroF1_std",
            "rank_test_score": "rank",
        }
    )


print("GroupKFold n_splits = 5, scoring = f1_macro")
print("groups = subjects_train_final (tik TRAIN)")

GroupKFold n_splits = 5, scoring = f1_macro
groups = subjects_train_final (tik TRAIN)


## Kas yra SVM ir ką daro C

**SVM** (Support Vector Machine) ieško hiperplokštumos, kuri klases atskiria su kuo didesne atsarga (margin). Daugiaklasiu atveju sklearn `LinearSVC` numatytai naudoja one-vs-rest: kiekvienai klasei mokomas linijinis sprendimo balas.

**C** — reguliarizacijos (kompromiso) parametras. Mažesnis C leidžia daugiau klaidų mokymo metu ir paprastesnę ribą; didesnis C stengiasi tiksliau atskirti TRAIN taškus, bet gali per daug prisitaikyti.

**Kodėl SVM reikia `StandardScaler`.** Linijinis SVM jautrus požymių masteliui: didesnės skalės požymiai stipriau veikia `w`. Standartizavimas visus požymius padaro panašaus masto.

**Kodėl scaleris Pipeline viduje.** Jei scalerį `fit` darytume visam TRAIN prieš CV, validavimo fold'o statistika nutekėtų į mokymą. Pipeline kiekviename fold'e scalerį moko tik iš CV mokymo dalies.

**Kodėl `GroupKFold` pagal žmogų.** Vieno žmogaus įrašai panašūs. Jei tas pats subject patektų ir į CV mokymą, ir į validavimą, CV balas būtų per geras ir neatspindėtų nematyto žmogaus.

**Kodėl TEST nenaudojamas paieškai.** Jei C rinktumėmės pagal TEST, TEST taptų slaptu mokymu ir galutinė metrika būtų pernelyg optimistiška.

## SVM hiperparametrų paieška tik TRAIN

`LinearSVC` Pipeline: `StandardScaler` + `LinearSVC`. Tinklelis: `C = [0.01, 0.1, 1, 10]`. `RANDOM_STATE=42` perduodamas `LinearSVC` (sklearn jį naudoja, kai dualus sprendimas maišo imtį). `max_iter=10000`, kad linijinis solveris spėtų konverguoti.

In [4]:
svm_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svm",
            LinearSVC(
                random_state=RANDOM_STATE,
                max_iter=10000,
                dual="auto",
            ),
        ),
    ]
)

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid={"svm__C": [0.01, 0.1, 1, 10]},
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

print("SVM GridSearchCV start (tik TRAIN, GroupKFold)")
fit_and_log_warnings(
    svm_grid,
    X_train,
    y_train,
    "SVM GridSearchCV",
    groups=subjects_train,
)

svm_cv_table = cv_summary(svm_grid, ["param_svm__C"]).rename(
    columns={"param_svm__C": "C"}
)
best_svm_C = float(svm_grid.best_params_["svm__C"])
best_svm_cv_macro_f1 = float(svm_grid.best_score_)

print()
print(svm_cv_table.to_string(index=False))
print()
print(f"Geriausias SVM C: {best_svm_C}")
print(f"SVM TRAIN CV Macro-F1 (vidurkis): {best_svm_cv_macro_f1:.6f}")

SVM GridSearchCV start (tik TRAIN, GroupKFold)
[SVM GridSearchCV] Warning nebuvo.

    C  CV_MacroF1_mean  CV_MacroF1_std  rank
 0.01         0.961494        0.016330     1
 0.10         0.959707        0.017832     2
 1.00         0.953066        0.019131     3
10.00         0.949684        0.021538     4

Geriausias SVM C: 0.01
SVM TRAIN CV Macro-F1 (vidurkis): 0.961494


## Galutinis SVM: mokymas visu TRAIN ir TEST įvertinimas

Po paieškos galutinį Pipeline su geriausiu C apmokome visu TRAIN ir tik tada prognozuojame TEST.

In [5]:
final_svm = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svm",
            LinearSVC(
                C=best_svm_C,
                random_state=RANDOM_STATE,
                max_iter=10000,
                dual="auto",
            ),
        ),
    ]
)

fit_and_log_warnings(final_svm, X_train, y_train, "galutinis SVM")
y_pred_svm = final_svm.predict(X_test)

svm_macro_f1 = f1_score(y_test, y_pred_svm, average="macro")
svm_bal_acc = balanced_accuracy_score(y_test, y_pred_svm)
svm_cm = confusion_df(y_test, y_pred_svm)

print(f"Galutinis SVM C={best_svm_C}")
print(f"SVM TEST Macro-F1: {svm_macro_f1:.6f}")
print(f"SVM TEST Balanced Accuracy: {svm_bal_acc:.6f}")
print()
print("SVM confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):")
print(svm_cm.to_string())

[galutinis SVM] Warning nebuvo.
Galutinis SVM C=0.01
SVM TEST Macro-F1: 0.897130
SVM TEST Balanced Accuracy: 0.893666

SVM confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):
                          pred_WALKING  pred_WALKING_UPSTAIRS  pred_WALKING_DOWNSTAIRS  pred_SITTING  pred_STANDING  pred_LAYING
tikra_WALKING                      505                      2                        0             1              0            0
tikra_WALKING_UPSTAIRS              10                    414                        1            59              0            0
tikra_WALKING_DOWNSTAIRS             3                     18                      414             1              0            0
tikra_SITTING                        0                      2                        0           472             70            0
tikra_STANDING                       0                      0                        0            97            487            0
tikra_LAYING                       

## SVM formulė ir ryšys su kodu

Kolokviume naudota linijinio daugiaklasio SVM taisyklė:

$$
f_k(x) = w_k^{T} x - b_k
$$

$$
\hat{y} = \arg\max_k f_k(x)
$$

- $x$ — požymių vektorius (čia: jau standartizuotas TEST įrašas);
- $w_k$ — $k$-osios klasės svorių vektorius;
- $b_k$ — $k$-osios klasės poslinkis (bias);
- $f_k(x)$ — klasės sprendimo balas;
- pasirenkama klasė, kurios balas didžiausias.

Žemiau naudojame **tą patį** jau apmokytą `final_svm`, o ne atskirą demonstracinį modelį.

Ką galima patikrinti iš realaus sklearn `LinearSVC`:
- `decision_function(X_test)` grąžina kiekvienos klasės balų matricą;
- `np.argmax(..., axis=1)` parenka didžiausio balo stulpelio indeksą;
- tas indeksas susiejamas su `classes_` (ne su 0..K-1 „iš galvos“, o su modelio klasių tvarka);
- sklearn viduje šie balai skaičiuojami kaip `X_scaled @ coef_.T + intercept_` — tai patikriname skaitmeniškai.

Kolokviumo formulėje poslinkis rašomas su minusu ($-b_k$). Sklearn `intercept_` yra pridėtinis narys (`+ intercept`). Jei $b_k = -\texttt{intercept}_k$, abi išraiškos reiškia tą patį balą. Čia **netvirtiname** daugiau, nei matome iš `coef_`, `intercept_` ir `decision_function`.

In [6]:
svm_step = final_svm.named_steps["svm"]
scaler_step = final_svm.named_steps["scaler"]

scores = final_svm.decision_function(X_test)
argmax_idx = np.argmax(scores, axis=1)
y_from_argmax = svm_step.classes_[argmax_idx]

X_test_scaled = scaler_step.transform(X_test)
scores_from_weights = X_test_scaled @ svm_step.coef_.T + svm_step.intercept_

print("LinearSVC classes_:", svm_step.classes_)
print("coef_ forma (klasės x požymiai):", svm_step.coef_.shape)
print("intercept_ forma:", svm_step.intercept_.shape)
print("decision_function(X_test) forma:", scores.shape)
print()
print("Pirmų 5 TEST įrašų balai (eilutė = įrašas, stulpelis = klasė pagal classes_):")
print(pd.DataFrame(scores[:5], columns=[id_to_name[int(c)] for c in svm_step.classes_]))
print()
print("argmax(axis=1) pirmi 5 indeksai:", argmax_idx[:5])
print("classes_[argmax] pirmos 5 klasės:", y_from_argmax[:5])
print("predict(X_test) pirmos 5 klasės:", y_pred_svm[:5])
print()

scores_match = np.allclose(scores, scores_from_weights)
pred_match = np.array_equal(y_from_argmax, y_pred_svm)
n_same = int(np.sum(y_from_argmax == y_pred_svm))

print(
    "decision_function sutampa su X_scaled @ coef_.T + intercept_:",
    scores_match,
)
print(
    f"argmax + classes_ sutampa su predict(): {pred_match} "
    f"({n_same} / {len(y_test)})"
)

assert scores_match, "decision_function turi sutapti su w^T x + intercept."
assert pred_match, "decision_function + argmax turi 100 % sutapti su predict()."
print("Patikra OK: formulės argmax taisyklė sutampa su galutinio SVM predict().")

LinearSVC classes_: [1 2 3 4 5 6]
coef_ forma (klasės x požymiai): (6, 561)
intercept_ forma: (6,)
decision_function(X_test) forma: (3155, 6)

Pirmų 5 TEST įrašų balai (eilutė = įrašas, stulpelis = klasė pagal classes_):
    WALKING  WALKING_UPSTAIRS  WALKING_DOWNSTAIRS   SITTING  STANDING  \
0 -1.079607         -1.773982           -1.485332  0.055874 -0.302043   
1 -1.061053         -1.757766           -1.310565 -1.628995  1.802855   
2 -0.975591         -1.555537           -1.466238 -0.905719  0.961111   
3 -1.148109         -1.400490           -1.438176 -1.419275  1.617864   
4 -0.858345         -1.455982           -1.418957 -0.816302  0.921483   

     LAYING  
0 -1.204382  
1 -1.220698  
2 -1.206319  
3 -1.338198  
4 -1.184008  

argmax(axis=1) pirmi 5 indeksai: [3 4 4 4 4]
classes_[argmax] pirmos 5 klasės: [4 5 5 5 5]
predict(X_test) pirmos 5 klasės: [4 5 5 5 5]

decision_function sutampa su X_scaled @ coef_.T + intercept_: True
argmax + classes_ sutampa su predict(): True (3155 

## Kas yra MLP, hidden_layer_sizes ir alpha

**MLP** (Multi-Layer Perceptron) — daugiasluoksnis perceptronas: įėjimas (561 požymis) eina per paslėptą sluoksnį (ar sluoksnius) su netiesine aktyvacija, tada į 6 klasių išėjimą.

**hidden_layer_sizes** — paslėptų sluoksnių dydis. `(50,)` reiškia vieną paslėptą sluoksnį su 50 neuronų, `(100,)` — vieną sluoksnį su 100 neuronų. Didesnis sluoksnis gali modeliuoti sudėtingesnius ryšius, bet brangiau mokosi ir lengviau pertreniuoja.

**alpha** — L2 reguliarizacijos stiprumas. Didesnė alpha baudžia didelius svorius ir paprastina tinklą.

Scaleris MLP reikalingas dėl tos pačios priežasties kaip SVM: gradientinis mokymas jautrus požymių masteliui. Jis vėl Pipeline viduje, kad CV metu nebūtų nutekėjimo. TEST vėl nenaudojamas nei paieškai, nei scaler `fit`.

## MLP hiperparametrų paieška tik TRAIN

Nedidelis tinklelis: `hidden_layer_sizes ∈ {(50,), (100,)}`, `alpha ∈ {0.0001, 0.001}`. `random_state=42`, `max_iter=500`. Jei atsiras `ConvergenceWarning`, jo neslepiame ir **nekeičiame** parametrų pagal TEST.

In [7]:
mlp_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                random_state=RANDOM_STATE,
                max_iter=500,
            ),
        ),
    ]
)

mlp_grid = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid={
        "mlp__hidden_layer_sizes": [(50,), (100,)],
        "mlp__alpha": [0.0001, 0.001],
    },
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

print("MLP GridSearchCV start (tik TRAIN, GroupKFold)")
fit_and_log_warnings(
    mlp_grid,
    X_train,
    y_train,
    "MLP GridSearchCV",
    groups=subjects_train,
)

mlp_cv_table = cv_summary(
    mlp_grid,
    ["param_mlp__hidden_layer_sizes", "param_mlp__alpha"],
).rename(
    columns={
        "param_mlp__hidden_layer_sizes": "hidden_layer_sizes",
        "param_mlp__alpha": "alpha",
    }
)
best_mlp_hidden = mlp_grid.best_params_["mlp__hidden_layer_sizes"]
best_mlp_alpha = float(mlp_grid.best_params_["mlp__alpha"])
best_mlp_cv_macro_f1 = float(mlp_grid.best_score_)

print()
print(mlp_cv_table.to_string(index=False))
print()
print(f"Geriausi MLP parametrai: hidden_layer_sizes={best_mlp_hidden}, alpha={best_mlp_alpha}")
print(f"MLP TRAIN CV Macro-F1 (vidurkis): {best_mlp_cv_macro_f1:.6f}")

MLP GridSearchCV start (tik TRAIN, GroupKFold)
[MLP GridSearchCV] Warning nebuvo.

hidden_layer_sizes  alpha  CV_MacroF1_mean  CV_MacroF1_std  rank
            (100,) 0.0010         0.956436        0.012435     1
            (100,) 0.0001         0.956299        0.012115     2
             (50,) 0.0010         0.952748        0.014239     3
             (50,) 0.0001         0.952004        0.014569     4

Geriausi MLP parametrai: hidden_layer_sizes=(100,), alpha=0.001
MLP TRAIN CV Macro-F1 (vidurkis): 0.956436


## Galutinis MLP: mokymas visu TRAIN ir TEST įvertinimas

In [8]:
final_mlp = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=best_mlp_hidden,
                alpha=best_mlp_alpha,
                random_state=RANDOM_STATE,
                max_iter=500,
            ),
        ),
    ]
)

fit_and_log_warnings(final_mlp, X_train, y_train, "galutinis MLP")
y_pred_mlp = final_mlp.predict(X_test)

mlp_step = final_mlp.named_steps["mlp"]
print(f"MLP n_iter_ = {mlp_step.n_iter_} (max_iter = {mlp_step.max_iter})")
if mlp_step.n_iter_ >= mlp_step.max_iter:
    print(
        "Pastaba: pasiektas max_iter, todėl sklearn gali rodyti ConvergenceWarning. "
        "TEST rezultatų dėl to nekeičiame ir hiperparametrų pagal TEST neperrenkame."
    )

mlp_macro_f1 = f1_score(y_test, y_pred_mlp, average="macro")
mlp_bal_acc = balanced_accuracy_score(y_test, y_pred_mlp)
mlp_cm = confusion_df(y_test, y_pred_mlp)

print()
print(
    f"Galutinis MLP: hidden_layer_sizes={best_mlp_hidden}, alpha={best_mlp_alpha}"
)
print(f"MLP TEST Macro-F1: {mlp_macro_f1:.6f}")
print(f"MLP TEST Balanced Accuracy: {mlp_bal_acc:.6f}")
print()
print("MLP confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):")
print(mlp_cm.to_string())

[galutinis MLP] Warning nebuvo.
MLP n_iter_ = 72 (max_iter = 500)

Galutinis MLP: hidden_layer_sizes=(100,), alpha=0.001
MLP TEST Macro-F1: 0.896616
MLP TEST Balanced Accuracy: 0.894957

MLP confusion matrix (eilutės = tikros klasės, stulpeliai = prognozės):
                          pred_WALKING  pred_WALKING_UPSTAIRS  pred_WALKING_DOWNSTAIRS  pred_SITTING  pred_STANDING  pred_LAYING
tikra_WALKING                      504                      2                        2             0              0            0
tikra_WALKING_UPSTAIRS              20                    461                        2             1              0            0
tikra_WALKING_DOWNSTAIRS            12                     43                      380             0              1            0
tikra_SITTING                        0                      1                        0           468             74            1
tikra_STANDING                       0                      0                        0          

## Rezultatų palyginimas

Žemiau dvi lentelės, kad nebūtų maišoma:

1. **TRAIN CV Macro-F1** — tik hiperparametrų parinkimui (TEST čia nenaudotas).
2. **TEST** — galutinis palyginimas tomis pačiomis sąlygomis. Majority Class ir k-NN eilutės paimtos iš `baseline_results.csv`, **neperskaičiuotos** kitu split.

Galutinės viso projekto išvados, kuris metodas geriausias, **nedarome**: dar nepadaryti abliacijos ir robustness eksperimentai. Čia tik objektyviai palyginamos šiame etape gautos TEST metrikos.

In [9]:
baseline_table = pd.read_csv(baseline_path)
print("Užkrauti baseline TEST rezultatai (neperskaičiuoti):")
print(baseline_table.to_string(index=False))
print()

cv_compare = pd.DataFrame(
    [
        {
            "Model": "SVM",
            "Pasirinkti parametrai": f"C={best_svm_C}",
            "TRAIN CV Macro-F1": best_svm_cv_macro_f1,
        },
        {
            "Model": "MLP",
            "Pasirinkti parametrai": (
                f"hidden_layer_sizes={best_mlp_hidden}, alpha={best_mlp_alpha}"
            ),
            "TRAIN CV Macro-F1": best_mlp_cv_macro_f1,
        },
    ]
)
print("Hiperparametrų parinkimas (tik TRAIN CV, metrika Macro-F1):")
print(cv_compare.to_string(index=False))
print()

stage3_test = pd.DataFrame(
    [
        {"Model": "SVM", "Macro-F1": svm_macro_f1, "Balanced Accuracy": svm_bal_acc},
        {"Model": "MLP", "Macro-F1": mlp_macro_f1, "Balanced Accuracy": mlp_bal_acc},
    ]
)
test_compare = pd.concat([baseline_table, stage3_test], ignore_index=True)
print("Galutinis TEST palyginimas:")
print(test_compare.to_string(index=False))
print()

if caught_warnings:
    print("Užfiksuoti warning (neslėpti):")
    print(pd.DataFrame(caught_warnings).drop_duplicates().to_string(index=False))
else:
    print("Warning šiame notebook paleidime nebuvo užfiksuota.")

Užkrauti baseline TEST rezultatai (neperskaičiuoti):
         Model  Macro-F1  Balanced Accuracy
Majority Class  0.053188           0.166667
          k-NN  0.844324           0.842074

Hiperparametrų parinkimas (tik TRAIN CV, metrika Macro-F1):
Model                  Pasirinkti parametrai  TRAIN CV Macro-F1
  SVM                                 C=0.01           0.961494
  MLP hidden_layer_sizes=(100,), alpha=0.001           0.956436

Galutinis TEST palyginimas:
         Model  Macro-F1  Balanced Accuracy
Majority Class  0.053188           0.166667
          k-NN  0.844324           0.842074
           SVM  0.897130           0.893666
           MLP  0.896616           0.894957

Warning šiame notebook paleidime nebuvo užfiksuota.


## Rezultatų išsaugojimas

`subject_disjoint_split.npz` ir `baseline_results.csv` **neperrašomi**.

In [10]:
results_dir.mkdir(parents=True, exist_ok=True)

stage3_path = results_dir / "intelligent_models_results.csv"
svm_cv_path = results_dir / "svm_cv_results.csv"
mlp_cv_path = results_dir / "mlp_cv_results.csv"
svm_cm_path = results_dir / "svm_confusion_matrix.csv"
mlp_cm_path = results_dir / "mlp_confusion_matrix.csv"
compare_path = results_dir / "test_comparison_stage3.csv"

stage3_test.to_csv(stage3_path, index=False)
svm_cv_table.to_csv(svm_cv_path, index=False)
mlp_cv_table.to_csv(mlp_cv_path, index=False)
svm_cm.to_csv(svm_cm_path)
mlp_cm.to_csv(mlp_cm_path)
test_compare.to_csv(compare_path, index=False)

print(f"Išsaugota: {stage3_path}")
print(f"Išsaugota: {svm_cv_path}")
print(f"Išsaugota: {mlp_cv_path}")
print(f"Išsaugota: {svm_cm_path}")
print(f"Išsaugota: {mlp_cm_path}")
print(f"Išsaugota: {compare_path}")
print("subject_disjoint_split.npz nekeistas.")
print("baseline_results.csv neperrašytas.")
print("3 etapas baigtas: SVM ir MLP įvertinti tuo pačiu split.")

Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\intelligent_models_results.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\svm_cv_results.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\mlp_cv_results.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\svm_confusion_matrix.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\mlp_confusion_matrix.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\test_comparison_stage3.csv
subject_disjoint_split.npz nekeistas.
baseline_results.c